### Reinforcement Learning and Multi-Armed Bandits

RL as a distinct learning paradigm from supervised learning, the multi-armed bandit problem as RL's simplest special case, epsilon-greedy exploration, and the on-policy/off-policy distinction. See `ab-testing.ipynb` for the connection: a bandit is essentially a continuously-adapting experiment, allocating more traffic to whichever arm looks best so far instead of a fixed 50/50 split for the whole run.

## RL vs. supervised learning

#### 0. Core idea

Supervised learning: every training example comes with a fixed, correct label, decided in advance, independent of what the model predicts. The loss is a direct comparison between prediction and label, and one example's label never depends on a decision made on a previous example.

Reinforcement learning: there's no fixed label, only a REWARD signal received after taking an action, and the action taken now can change which states and rewards you'll even get to observe later (the data distribution is a consequence of your own past decisions, not fixed in advance). This is what makes RL both harder and necessary for problems like bandits and recommender ranking (`recommender-systems.ipynb`) under exploration, where showing a user item X instead of item Y doesn't just cost you one label, it changes what you'll ever learn about item Y for that user.

## The multi-armed bandit problem and exploration-exploitation

#### 0. Core idea

A bandit has K arms (options), each with an unknown, fixed probability of paying out a reward. Each round you pick one arm, observe its reward, and want to maximize cumulative reward over time. The core tension: **exploit** the arm that currently looks best (based on the limited data you have so far), or **explore** a less-tried arm that might actually be better but you don't have enough data yet to know.

Toy setup, 3 arms with hidden true reward probabilities `A=0.3, B=0.5, C=0.2` (unknown to the algorithm, only used here to generate rewards and check performance).

## Epsilon-greedy

#### 0. Core idea

Simplest fix for the exploration-exploitation tradeoff: with probability epsilon, pick a uniformly random arm (explore); otherwise pick the arm with the highest observed average reward so far (exploit). Pure greedy (`epsilon=0`) risks locking onto a suboptimal arm forever if its first few samples happen to look good by chance, since it then never revisits the other arms to find out it was wrong.

In [ ]:
import numpy as np

true_probs = {"A": 0.3, "B": 0.5, "C": 0.2}  # B is the actual best arm
arms = list(true_probs.keys())

def run_epsilon_greedy(epsilon, n_rounds, seed):
    rng = np.random.default_rng(seed)
    counts = {a: 0 for a in arms}
    sums = {a: 0.0 for a in arms}
    total_reward = 0
    for t in range(n_rounds):
        if t < len(arms):
            a = arms[t]  # try each arm once first
        elif rng.random() < epsilon:
            a = arms[rng.integers(len(arms))]
        else:
            avg = {k: (sums[k] / counts[k] if counts[k] > 0 else 0) for k in arms}
            a = max(avg, key=avg.get)
        r = 1 if rng.random() < true_probs[a] else 0
        counts[a] += 1
        sums[a] += r
        total_reward += r
    return total_reward, counts

for eps in [0.0, 0.1]:
    total_reward, counts = run_epsilon_greedy(eps, n_rounds=300, seed=3)
    print(f"epsilon={eps}: total_reward={total_reward}, arm_counts={counts}, reward_rate={round(total_reward/300, 3)}")

#### 1. Reading the result

With this particular random seed, pure greedy (`epsilon=0`) gets unlucky on its very first pass through the arms, arm A's single initial sample happens to pay off, so greedy locks onto A and plays it 295 of 300 rounds, converging to A's true reward rate of ~0.30 and never discovering that B (true rate 0.50) is actually better. Epsilon-greedy with `epsilon=0.1`, given the exact same underlying luck, still spends enough random rounds re-sampling every arm that it eventually notices B is better and shifts most of its plays there (243 of 300), nearly doubling the reward rate to ~0.50. This is the concrete cost of pure exploitation: a small, fixed amount of continued exploration is what lets the algorithm recover from a bad first impression.

## On-policy vs. off-policy

#### 0. Core idea

**On-policy**: evaluate or improve the SAME policy that's generating the data (e.g. epsilon-greedy above learns directly from the actions epsilon-greedy itself chose). **Off-policy**: learn about a different, TARGET policy using data generated by a different BEHAVIOR policy (e.g. estimate how a proposed new ranking model would have performed, using only logged data from the current production model). Off-policy evaluation needs a correction, since the behavior policy over-represents the actions it liked, using an action the target policy favors much more than the behavior policy did should count for more, not less.

Importance sampling is the standard correction: reweight each observed reward by `target_policy_prob(action) / behavior_policy_prob(action)`. Toy: the behavior policy took a specific action with probability 0.2, but the target policy under evaluation would have taken that same action with probability 0.5, and the observed reward was 10.

In [ ]:
behavior_prob = 0.2
target_prob = 0.5
observed_reward = 10

importance_weight = target_prob / behavior_prob
weighted_reward = observed_reward * importance_weight
print("importance weight:", importance_weight)
print("importance-weighted reward estimate:", weighted_reward)

#### 1. Why this matters for offline evaluation

The naive estimate would just use the raw observed reward of 10, but that under-counts this action's true value under the target policy, which would have chosen it far more often (0.5 vs. 0.2). The importance-weighted estimate of 25 corrects for that mismatch. In practice this is exactly why off-policy evaluation from historical logs is a viable, much cheaper alternative to running a full new online experiment (`ab-testing.ipynb`) every time you want to test a new ranking or bandit policy, at the cost of high variance when the behavior and target policies diverge a lot (very small `behavior_prob` values blow the importance weight up).